# 00 · Configuración común

Este notebook **no se ejecuta solo**: los notebooks 01–05 lo cargan al inicio con `%run ./00_configuracion.ipynb`.
Centraliza en un único lugar lo que todos comparten, para que un cambio (semilla, conjunto de predictores, paleta) se
propague a todo el análisis:

- rutas del proyecto, semilla y parámetros de validación;
- diccionario de variables con la codificación oficial de UCI (`data/raw/heart-disease.names`);
- conjuntos de predictores de cada modelo;
- paleta de colores validada y estilo de las figuras;
- utilidades compartidas (formato de valores p, IC de Wilson, matriz de diseño, exportación a LaTeX/JSON).

In [ ]:
import hashlib
import json
import time
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display, Markdown
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## Rutas, reproducibilidad, diccionario de variables y conjuntos de predictores

In [ ]:
# --------------------------------------------------------------------------
# Rutas
# --------------------------------------------------------------------------
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
TABLES = RESULTS / "tables"

UCI_FILE = DATA_RAW / "processed.cleveland.data"
UCI_SHA256 = "a74b7efa387bc9d108d7d0115d831fe9b414b29ae7124f331b622b4efa0427c8"
KAGGLE_FILE = DATA_RAW / "heart_disease_kaggle.csv"

# --------------------------------------------------------------------------
# Reproducibilidad
# --------------------------------------------------------------------------
SEED = 2026
CV_FOLDS = 5
CV_REPEATS = 20          # 5 x 20 = 100 particiones de validación cruzada
N_BOOTSTRAP = 500        # bootstrap de optimismo (Harrell) e IC
ALPHA = 0.05
TARGET_SENSITIVITY = 0.90  # umbral orientado a tamizaje

# --------------------------------------------------------------------------
# Diccionario de variables (codificación oficial UCI, heart-disease.names)
# --------------------------------------------------------------------------
UCI_COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
               "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]

VARIABLES = {
    "age":      dict(etiqueta="Edad", unidad="años", tipo="cuantitativa continua",
                     escala="razón", rol="independiente", grupo="clínica"),
    "sex":      dict(etiqueta="Sexo masculino", unidad="", tipo="cualitativa dicotómica",
                     escala="nominal", rol="independiente", grupo="clínica",
                     categorias={0: "Mujer", 1: "Hombre"}),
    "cp":       dict(etiqueta="Tipo de dolor torácico", unidad="", tipo="cualitativa politómica",
                     escala="nominal", rol="independiente", grupo="clínica",
                     categorias={1: "Angina típica", 2: "Angina atípica",
                                 3: "Dolor no anginoso", 4: "Asintomático"}),
    "trestbps": dict(etiqueta="Presión arterial sistólica en reposo", unidad="mmHg",
                     tipo="cuantitativa continua", escala="razón", rol="independiente",
                     grupo="clínica"),
    "chol":     dict(etiqueta="Colesterol sérico total", unidad="mg/dL",
                     tipo="cuantitativa continua", escala="razón", rol="independiente",
                     grupo="laboratorio"),
    "fbs":      dict(etiqueta="Glucemia en ayunas > 120 mg/dL", unidad="",
                     tipo="cualitativa dicotómica", escala="nominal", rol="independiente",
                     grupo="laboratorio", categorias={0: "No", 1: "Sí"}),
    "restecg":  dict(etiqueta="ECG en reposo", unidad="", tipo="cualitativa politómica",
                     escala="nominal", rol="independiente", grupo="ECG reposo",
                     categorias={0: "Normal", 1: "Anomalía ST-T", 2: "Hipertrofia VI"}),
    "restecg_anormal": dict(etiqueta="ECG en reposo anormal (ST-T o HVI)", unidad="",
                     tipo="cualitativa dicotómica (derivada)", escala="nominal",
                     rol="independiente", grupo="ECG reposo", categorias={0: "No", 1: "Sí"}),
    "thalach":  dict(etiqueta="Frecuencia cardíaca máxima alcanzada", unidad="lpm",
                     tipo="cuantitativa continua", escala="razón", rol="independiente",
                     grupo="prueba de esfuerzo"),
    "exang":    dict(etiqueta="Angina inducida por ejercicio", unidad="",
                     tipo="cualitativa dicotómica", escala="nominal", rol="independiente",
                     grupo="prueba de esfuerzo", categorias={0: "No", 1: "Sí"}),
    "oldpeak":  dict(etiqueta="Depresión del ST inducida por ejercicio", unidad="mm",
                     tipo="cuantitativa continua", escala="razón", rol="independiente",
                     grupo="prueba de esfuerzo"),
    "slope":    dict(etiqueta="Pendiente del ST en el pico de ejercicio", unidad="",
                     tipo="cualitativa politómica", escala="ordinal", rol="independiente",
                     grupo="prueba de esfuerzo",
                     categorias={1: "Ascendente", 2: "Plana", 3: "Descendente"}),
    "ca":       dict(etiqueta="Vasos mayores coloreados por fluoroscopia", unidad="n.º (0-3)",
                     tipo="cuantitativa discreta", escala="razón", rol="independiente",
                     grupo="imagen"),
    "thal":     dict(etiqueta="Gammagrafía con talio", unidad="", tipo="cualitativa politómica",
                     escala="nominal", rol="independiente", grupo="imagen",
                     categorias={3: "Normal", 6: "Defecto fijo", 7: "Defecto reversible"}),
    "ec":       dict(etiqueta="Enfermedad coronaria angiográfica (estenosis > 50 %)",
                     unidad="", tipo="cualitativa dicotómica", escala="nominal",
                     rol="dependiente (desenlace)", grupo="desenlace",
                     categorias={0: "Sin EC", 1: "Con EC"}),
}

CONTINUOUS = ["age", "trestbps", "chol", "thalach", "oldpeak"]
DISCRETE = ["ca"]
BINARY = ["sex", "fbs", "exang"]
DERIVED_BINARY = ["restecg_anormal"]  # solo en modelos
CATEGORICAL = ["cp", "restecg", "slope", "thal"]

# Categoría de referencia para OR (la de menor riesgo clínico esperado)
REFERENCE = {"cp": 1, "restecg": 0, "slope": 1, "thal": 3}

# Unidades clínicamente interpretables para OR de variables continuas
OR_UNITS = {"age": 10, "trestbps": 10, "chol": 40, "thalach": 10, "oldpeak": 1, "ca": 1}

# --------------------------------------------------------------------------
# Conjuntos de predictores (pregunta central del proyecto)
# --------------------------------------------------------------------------
# Modelo básico: solo información de consulta, ECG de reposo y prueba de esfuerzo
# (bajo costo, no invasiva, disponible en primer nivel / consulta externa).
# restecg se colapsa a binaria (anomalía ST-T tiene n = 4 -> estimación inestable).
PRED_BASICO = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg_anormal",
               "thalach", "exang", "oldpeak", "slope"]
# Modelo extendido: añade estudios de imagen (fluoroscopia y gammagrafía con talio).
PRED_EXTENDIDO = PRED_BASICO + ["ca", "thal"]
# Réplica del modelo del semestre 1 (4 variables continuas) como línea base histórica.
PRED_SEMESTRE1 = ["age", "chol", "thalach", "trestbps"]

# --------------------------------------------------------------------------
# Paleta (validada: dataviz reference palette, modo claro)
# --------------------------------------------------------------------------
COLORS = {
    "sin_ec": "#2a78d6",      # slot 1 azul
    "con_ec": "#eb6834",      # slot 2 naranja
    "m_basico": "#2a78d6",
    "m_extendido": "#eb6834",
    "m_arbol": "#1baf7a",     # slot 3 aqua
    "m_rf": "#6250d6",        # slot 7 violeta (solo en tablas / distribución)
    "m_s1": "#898781",        # gris de referencia
    "crudo": "#898781",
    "surface": "#fcfcfb",
    "ink": "#0b0b0b",
    "ink2": "#52514e",
    "muted": "#898781",
    "grid": "#e1e0d9",
    "axis": "#c3c2b7",
    "div_neg": "#2a78d6",
    "div_mid": "#f0efec",
    "div_pos": "#e34948",
}

# --------------------------------------------------------------------------
# Definiciones conceptual (qué es) y operacional (cómo se midió en la base)
# --------------------------------------------------------------------------
DEFINICIONES = {
    "age": ("Tiempo de vida del paciente.", "Años cumplidos al momento del estudio."),
    "sex": ("Sexo biológico.", "1 = hombre, 0 = mujer, según el registro clínico."),
    "cp": ("Caracterización clínica del dolor torácico según su relación con el esfuerzo y su respuesta a nitratos.",
           "Clasificación del médico tratante: típica (cumple 3 criterios), atípica (2), no anginoso (≤ 1), asintomático."),
    "trestbps": ("Presión que ejerce la sangre sobre las arterias durante la sístole, en reposo.",
                 "Presión sistólica en mmHg medida al ingreso hospitalario."),
    "chol": ("Concentración de colesterol total en el suero.", "mg/dL en sangre periférica."),
    "fbs": ("Hiperglucemia en ayunas, marcador de alteración del metabolismo de la glucosa.",
            "1 si la glucemia en ayunas es > 120 mg/dL; 0 en caso contrario."),
    "restecg": ("Hallazgos del electrocardiograma de 12 derivaciones en reposo.",
                "0 normal; 1 anomalía ST-T (inversión de T o elevación/depresión del ST > 0,05 mV); "
                "2 hipertrofia ventricular izquierda probable o definitiva (criterios de Estes)."),
    "restecg_anormal": ("ECG de reposo con cualquier anormalidad.", "1 si restecg ∈ {1, 2}; 0 si es normal."),
    "thalach": ("Respuesta cronotrópica al ejercicio.", "Frecuencia cardíaca máxima (lpm) alcanzada en la prueba de esfuerzo."),
    "exang": ("Isquemia miocárdica sintomática provocada por el esfuerzo.",
              "1 si hubo angina durante la prueba de esfuerzo; 0 si no."),
    "oldpeak": ("Isquemia miocárdica electrocardiográfica provocada por el esfuerzo.",
                "Depresión del segmento ST (mm) inducida por ejercicio, relativa al reposo."),
    "slope": ("Morfología del segmento ST en el pico del ejercicio.", "1 ascendente, 2 plana, 3 descendente."),
    "ca": ("Calcificación de arterias coronarias principales, marcador de aterosclerosis.",
           "Número de vasos mayores (0–3) coloreados en la fluoroscopia."),
    "thal": ("Perfusión miocárdica evaluada con gammagrafía con talio-201.",
             "3 normal; 6 defecto fijo (cicatriz); 7 defecto reversible (isquemia inducible)."),
    "ec": ("Enfermedad coronaria aterosclerótica significativa.",
           "Estenosis > 50 % del diámetro en al menos un vaso mayor en la angiografía coronaria (num ≥ 1)."),
}

In [ ]:
for _d in (DATA_PROCESSED, FIGURES, TABLES, RESULTS / "json"):
    _d.mkdir(parents=True, exist_ok=True)

## Estilo de figuras (paleta validada, 300 dpi)

In [ ]:
K = COLORS
GROUP_COLORS = [K["sin_ec"], K["con_ec"]]
GROUP_LABELS = ["Sin EC", "Con EC"]

def setup():
    plt.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
        "figure.facecolor": K["surface"], "axes.facecolor": K["surface"],
        "savefig.facecolor": K["surface"],
        "font.family": "DejaVu Sans", "font.size": 9,
        "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "axes.labelsize": 9, "axes.labelcolor": K["ink2"],
        "axes.edgecolor": K["axis"], "axes.linewidth": 0.8,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": K["grid"], "grid.linewidth": 0.6,
        "axes.axisbelow": True,
        "xtick.color": K["muted"], "ytick.color": K["muted"],
        "xtick.labelcolor": K["ink2"], "ytick.labelcolor": K["ink2"],
        "legend.frameon": False, "legend.fontsize": 8,
        "text.color": K["ink"], "lines.linewidth": 2,
    })


def save(fig, name):
    FIGURES.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES / f"{name}.png")
    display(fig)  # se muestra en el notebook y queda guardada en results/figures
    plt.close(fig)


def _box_strip(ax, groups, colors, labels, rng):
    bp = ax.boxplot(groups, widths=0.45, patch_artist=True, showfliers=False,
                    medianprops=dict(color=K["ink"], linewidth=1.5),
                    whiskerprops=dict(color=K["muted"], linewidth=1),
                    capprops=dict(color=K["muted"], linewidth=1),
                    boxprops=dict(linewidth=1))
    for patch, col in zip(bp["boxes"], colors):
        patch.set_facecolor(col + "33"); patch.set_edgecolor(col)
    for i, (g, col) in enumerate(zip(groups, colors), start=1):
        x = i + rng.uniform(-0.17, 0.17, len(g))
        ax.scatter(x, g, s=9, color=col, alpha=0.55, linewidths=0, zorder=3)
    ax.set_xticks(range(1, len(groups) + 1), labels)
    ax.grid(axis="x", visible=False)


TREE_NAMES = {
    "age": "Edad (años)", "sex": "Sexo masculino", "trestbps": "PAS reposo (mmHg)",
    "chol": "Colesterol (mg/dL)", "fbs": "Glucemia > 120", "restecg_anormal": "ECG reposo anormal",
    "thalach": "FC máxima (lpm)", "exang": "Angina de esfuerzo", "oldpeak": "Depresión ST (mm)",
    "ca": "Vasos en fluoroscopia", "cp_2": "Dolor: angina atípica", "cp_3": "Dolor: no anginoso",
    "cp_4": "Dolor: asintomático", "slope_2": "Pendiente ST plana", "slope_3": "Pendiente ST descendente",
    "thal_6": "Talio: defecto fijo", "thal_7": "Talio: defecto reversible",
}

## Utilidades compartidas

In [ ]:
def fmt_p(p):
    if pd.isna(p):
        return ""
    return "< 0,001" if p < 0.001 else f"{p:.3f}".replace(".", ",")


def wilson(k, n):
    lo, hi = proportion_confint(k, n, alpha=ALPHA, method="wilson")
    return lo, hi


def design_matrix(df, predictors):
    """Matriz de diseño con indicadoras (categoría de referencia en config.REFERENCE)."""
    X = pd.DataFrame(index=df.index)
    for v in predictors:
        if v in CATEGORICAL:
            cats = sorted(int(c) for c in df[v].dropna().unique())
            ref = REFERENCE[v]
            for c in cats:
                if c != ref:
                    X[f"{v}={c}"] = (df[v] == c).astype(float)
        else:
            X[v] = df[v].astype(float)
    return X


def term_label(term):
    if "=" in term:
        v, c = term.split("=")
        meta = VARIABLES[v]
        ref = meta["categorias"][REFERENCE[v]]
        return f"{meta['etiqueta']}: {meta['categorias'][int(c)]} (vs. {ref.lower()})"
    meta = VARIABLES[term]
    if term in BINARY or term in DERIVED_BINARY:
        return meta["etiqueta"]
    if term == "ca":
        return f"{meta['etiqueta']} (por cada vaso)"
    unit = OR_UNITS.get(term, 1)
    return f"{meta['etiqueta']} (por {unit} {meta['unidad']})".replace("  ", " ")

In [ ]:
def to_latex(df, path, caption, label, cols=None, header=None):
    d = df[cols] if cols else df
    if header:
        d = d.rename(columns=dict(zip(d.columns, header)))
    tex = d.to_latex(index=False, escape=True, caption=caption, label=label,
                     position="H", column_format="l" + "r" * (d.shape[1] - 1))
    path.write_text(tex, encoding="utf-8")


def fmt_or(t):
    t = t.copy()
    t["OR (IC 95 %)"] = [f"{o:.2f} ({lo:.2f}–{hi:.2f})".replace(".", ",")
                         for o, lo, hi in zip(t.OR, t.IC95_inf, t.IC95_sup)]
    t["p_fmt"] = t.p.map(fmt_p)
    return t


def save_json(name, obj):
    """Guarda resultados parciales en results/json/<name>.json (el notebook 05 los integra)."""
    path = RESULTS / "json" / f"{name}.json"
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2,
                               default=lambda o: o.item() if hasattr(o, "item") else str(o)), encoding="utf-8")
    return path


def load_json(name):
    return json.loads((RESULTS / "json" / f"{name}.json").read_text(encoding="utf-8"))


def load_processed():
    """Base depurada (generada por 01). Devuelve (df con n = 303, cc con casos completos)."""
    df = pd.read_csv(DATA_PROCESSED / "cleveland_depurada.csv")
    df["ca"] = df["ca"].astype("Int64"); df["thal"] = df["thal"].astype("Int64")
    cc = df[df.completo].copy()
    cc["ca"] = cc["ca"].astype(int); cc["thal"] = cc["thal"].astype(int)
    return df, cc